In [ ]:
# Baseline: TF-IDF + LogisticRegression
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib

RANDOM_SEED = 42

# load
train = pd.read_csv('/Users/maximus/Downloads/SCHOOL/bt4012/bt4012-competition-2/data/raw/train.csv')   # must contain cols: id, text, label
# train = pd.read_csv('/Users/maximus/Downloads/SCHOOL/bt4012/bt4012-competition-2/data/processed/train_features.csv')
test  = pd.read_csv('/Users/maximus/Downloads/SCHOOL/bt4012/bt4012-competition-2/data/raw/test.csv')    # must contain cols: id, text

# quick clean (minimal)
train['text'] = train['text'].astype(str)
test['text']  = test['text'].astype(str)

# vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    max_features=50000,
    strip_accents='unicode',
    lowercase=True
)
X = vectorizer.fit_transform(train['text'])
y = train['label'].values
X_test = vectorizer.transform(test['text'])

# model
clf = LogisticRegression(
    solver='saga',
    C=1.0,
    penalty='l2',
    max_iter=2000,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

# cross-validate
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
accs = cross_val_score(clf, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
f1s  = cross_val_score(clf, X, y, cv=skf, scoring='f1', n_jobs=-1)
aucs = cross_val_score(clf, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
print('CV accuracy:', accs.mean(), '±', accs.std())
print('CV f1      :', f1s.mean(), '±', f1s.std())
print('CV AUC     :', aucs.mean(), '±', aucs.std())

# train on full data and predict
clf.fit(X, y)
preds_test = clf.predict(X_test)
probs_test = clf.predict_proba(X_test)[:,1]

# save model + vectorizer
joblib.dump({'vectorizer': vectorizer, 'model': clf}, 'baseline_tfidf_logreg.joblib')

# make submission with probabilities
sub = pd.DataFrame({'id': test['id'], 'target': probs_test})
sub.to_csv('submission_baseline.csv', index=False)


CV accuracy: 0.9323010314013415 ± 0.0035018015013585017
CV f1      : 0.8805624761721507 ± 0.0062420027297240286
CV AUC     : 0.9933066552440664 ± 0.0009193365118554324


In [ ]:
# Baseline: TF-IDF + LogisticRegression
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline

RANDOM_SEED = 42

# load
train = pd.read_csv('/Users/maximus/Downloads/SCHOOL/bt4012/bt4012-competition-2/data/raw/train.csv')
test  = pd.read_csv('/Users/maximus/Downloads/SCHOOL/bt4012/bt4012-competition-2/data/raw/test.csv')

# quick clean (minimal)
train['text'] = train['text'].astype(str)
test['text']  = test['text'].astype(str)

# vectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=3,
    max_df=0.95,
    max_features=50000,
    strip_accents='unicode',
    lowercase=True
)
X = vectorizer.fit_transform(train['text'])
y = train['label'].values
X_test = vectorizer.transform(test['text'])

# Create pipeline with SVD
svd = TruncatedSVD(n_components=1000, random_state=RANDOM_SEED)
clf_with_svd = Pipeline([
    ('svd', svd),
    ('classifier', LogisticRegression(
        solver='saga',
        C=1.0,
        penalty='l2',
        max_iter=2000,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ))
])

# cross-validate
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
accs = cross_val_score(clf_with_svd, X, y, cv=skf, scoring='accuracy', n_jobs=-1)
f1s  = cross_val_score(clf_with_svd, X, y, cv=skf, scoring='f1', n_jobs=-1)
aucs = cross_val_score(clf_with_svd, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)
print('CV accuracy:', accs.mean(), '±', accs.std())
print('CV f1      :', f1s.mean(), '±', f1s.std())
print('CV AUC     :', aucs.mean(), '±', aucs.std())

# train on full data and predict
clf_with_svd.fit(X, y)
preds_test = clf_with_svd.predict(X_test)
probs_test = clf_with_svd.predict_proba(X_test)[:,1]

# save model + vectorizer
joblib.dump({'vectorizer': vectorizer, 'model': clf_with_svd}, 'baseline_tfidf_logreg.joblib')

# make submission with probabilities
sub = pd.DataFrame({'id': test['id'], 'target': probs_test})
sub.to_csv('submission_baseline.csv', index=False)


KeyError: 'text'